# Lab 03: The ReAct Pattern for Step-by-Step Verification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab03_react_pattern/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

Lab 02 let the model call tools. **ReAct** adds the missing half: the model says *why* it is
calling each tool before it calls it, reads the result, and only then decides the next step. The
loop is **Thought -> Action -> Observation**, repeated until the evidence is enough to answer.

You will answer one narrow timing question: *did someone try to send an image through Signal while a
phone was left unattended, and what can the network evidence tell us about delivery?* You will walk
the loop **one turn at a time** (Part A), then run the same case through a packaged `ReactAgent`
(Part B). The agent is a bounded aid; the final judgment stays with you.

## Part 0: One-Time Setup

Run these cells in order. They install one library, connect you to a **free hosted model** (Groq),
write the case files, and load the ReAct framework. No local install is required.

### Step 0.1: Install the client library

**Purpose:** install the `openai` client, the one small library that lets the notebook talk to the model (Groq uses the same interface).  
**What you'll see:** a short pip log. Warnings are normal.

In [ ]:
# Purpose: install the one library this notebook needs, inside the hosted session only.
%pip install -q openai

### Step 0.2: Connect to a free hosted model (Groq)

**Purpose:** get a free **API key** (a private password that lets the notebook use the model) and build a client.

1. Open **https://console.groq.com/keys**, sign in, and create a key.
2. Run the cell and paste the key when asked. It is hidden and is **not saved** into the notebook.

**What you'll see:** `Model OK: ...` once the connection works.

In [ ]:
# Purpose: collect the Groq key privately and build an OpenAI-compatible client pointed at Groq.
import os        # os: lets the notebook read saved settings, such as where your key is kept
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: the client that carries your messages to the model (Groq speaks the same language)

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

BASE_URL = 'https://api.groq.com/openai/v1'
# Free models that work well for this lab, best first. Groq retires models from time to time, so
# this cell uses the first one your key can reach and prints which. Edit the list to try another.
# Qwen comes first here: this lab's agent writes <tool_call> tags as plain text, which Qwen follows;
# gpt-oss tends to plan the call silently and return an empty reply.
PREFERRED = ['qwen/qwen3.8-27b', 'openai/gpt-oss-120b']

# max_retries=8: the free tier allows about 8,000 tokens a minute. When a call goes over, Groq says
# 'try again in a few seconds' and the client waits and retries instead of stopping with an error.
client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'], max_retries=8)

# Every model call in this notebook goes through this small wrapper. It does two things:
# 1. Groq counts each request's prompt PLUS its reply limit against that budget, so each call gets a
#    reply limit: 1,500 tokens, or 900 for Qwen, whose free tier allows 1,000 reply tokens a
#    minute (any call can still set its own max_tokens).
# 2. gpt-oss sometimes tries a built-in function call instead of writing text, which Groq rejects
#    ('model called a tool'). The wrapper then asks again with a plain-text reminder.
_create = client.chat.completions.create
def _create_within_limits(*args, **kw):
    kw.setdefault('max_tokens', 900 if 'qwen' in str(kw.get('model')) else 1500)
    for attempt in range(3):
        try:
            return _create(*args, **kw)
        except Exception as err:
            if 'called a tool' not in str(err) or attempt == 2:
                raise
            kw['messages'] = list(kw.get('messages', [])) + [
                {'role': 'user', 'content': 'Reply in plain text only; do not call functions.'}]
client.chat.completions.create = _create_within_limits

available = [m.id for m in client.models.list().data]
MODEL = next((m for m in PREFERRED if m in available), None)
if MODEL is None:
    print('None of PREFERRED is on your key right now. Models your key can use:')
    for m in available:
        print('  -', m)
    raise SystemExit('Put one of these chat models first in PREFERRED and re-run this cell.')

probe = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
print('Model OK:', MODEL, '->', (probe.choices[0].message.content or '').strip())

### Step 0.3: Write the synthetic case files

**Purpose:** create a local `data/` folder with the five artifacts for case `RA-2026-009`. These are
synthetic and identical to the repository's copies.  
**What you'll see:** the files written. Existing files are not overwritten.

In [ ]:
# Purpose: create the case files on disk so the notebook has everything it needs, with no download.
from pathlib import Path  # Path: a readable way to refer to files and folders by name

ARTIFACTS = {'artifact_manifest.json': '{\n  "case_id": "RA-2026-009",\n  "lab": "Lab 03 - ReAct Pattern",\n  "device": {\n    "platform": "Android",\n    "model": "Samsung Galaxy A54",\n    "os_version": "Android 14"\n  },\n  "acquisition": {\n    "type": "full_file_system",\n    "tool": "Cellebrite UFED 7.70",\n    "acquired_at_utc": "2026-02-21T10:43:18Z",\n    "original_image_sha256": "a7a4db1c2e1d75889f847d0b46a60bf8b84ae4d1533f6f5ce7aa98fb29dcb6ef"\n  },\n  "analysis_timezone": "America/New_York",\n  "incident_window_utc": {\n    "start": "2026-02-20T14:10:00Z",\n    "end": "2026-02-20T14:25:00Z"\n  },\n  "interval_source": "staff observation log",\n  "note": "Synthetic classroom dataset for react-pattern demo built around a reported unattended interval"\n}\n', 'incident_window.csv': 'timestamp_utc,event,source,notes\n2026-02-20T14:10:00Z,unattended_interval_start,staff_observation,phone left on supply table before coordinator stepped away\n2026-02-20T14:25:00Z,unattended_interval_end,staff_observation,phone recovered from supply table when coordinator returned\n', 'messaging_events.csv': 'timestamp_utc,app,event,contact,details\n2026-02-20T14:16:11Z,Signal,attachment_attempt,+1-555-0179,outgoing image attachment attempt recorded while mobile data unavailable\n', 'network_events.csv': 'timestamp_utc,event,status,details\n2026-02-20T14:15:58Z,connectivity_change,offline,mobile data dropped during reported unattended interval\n2026-02-20T14:28:02Z,connectivity_change,restored,mobile data restored after reported unattended interval\n', 'chain_of_custody.csv': 'timestamp_utc,handler,action,notes\n2026-02-21T10:43:18Z,Analyst Rivera,acquisition,full file system collected from outreach phone\n2026-02-21T11:05:44Z,Analyst Rivera,hash_verification,original image hash verified before review\n'}

data_dir = Path('data'); data_dir.mkdir(exist_ok=True)
for name, text in ARTIFACTS.items():
    fp = data_dir / name
    if not fp.exists():
        fp.write_text(text)
    print(('wrote ' if fp.stat().st_size else 'exists ') + str(fp))

### Step 0.4: Load the ReAct framework

**Purpose:** load the tool wrapper (`@tool`), the `extract_tag_content` helper that reads the model's
`<thought>`, `<tool_call>`, and `<response>` tags, and the packaged `ReactAgent`. This is a self-contained
copy adapted from The Neural Maze's agentic-patterns-course (MIT License).  
**What you'll see:** nothing printed; it just defines the classes used below.

In [ ]:
# Purpose: the ReAct framework (@tool, extract_tag_content, ReactAgent). Self-contained copy for this notebook.
# Adapted from The Neural Maze agentic-patterns-course (MIT License), (c) 2024 The Neural Maze.

import json
import re
from dataclasses import dataclass
from typing import Callable


# ---------------------------------------------------------------- tag extraction
@dataclass
class TagContentResult:
    content: list
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """Extract everything enclosed by <tag>...</tag> (used for <thought>, <tool_call>, <response>)."""
    if tag == "tool_call":
        text = re.sub(r"(</tool_call>\s*){2,}$", "</tool_call>", text.strip())
    pattern = rf"<{tag}>(.*?)</{tag}>"
    matches = re.findall(pattern, text, re.DOTALL)
    return TagContentResult(content=[m.strip() for m in matches], found=bool(matches))


# ---------------------------------------------------------------- chat history
def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


def update_chat_history(history: list, msg: str, role: str):
    history.append(build_prompt_structure(prompt=msg, role=role))


class ChatHistory(list):
    def __init__(self, messages=None, total_length=-1):
        super().__init__(messages or [])
        self.total_length = total_length

    def append(self, msg):
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)


def completions_create(client, messages, model) -> str:
    response = client.chat.completions.create(messages=list(messages), model=model)
    return str(response.choices[0].message.content)


# ---------------------------------------------------------------- tool wrapper
def get_fn_signature(fn: Callable) -> dict:
    sig = {"name": fn.__name__, "description": fn.__doc__, "parameters": {"properties": {}}}
    schema = {k: {"type": v.__name__} for k, v in fn.__annotations__.items() if k != "return"}
    sig["parameters"]["properties"] = schema
    return sig


def validate_arguments(tool_call: dict, tool_signature: dict) -> dict:
    properties = tool_signature["parameters"]["properties"]
    type_mapping = {"int": int, "str": str, "bool": bool, "float": float}
    validated = {}
    for name, value in tool_call["arguments"].items():
        schema = properties.get(name)
        if schema is None:
            continue
        converter = type_mapping.get(schema.get("type"))
        if converter is None:
            validated[name] = value
            continue
        if not isinstance(value, converter):
            value = converter(value)
        validated[name] = value
    tool_call["arguments"] = validated
    return tool_call


class Tool:
    def __init__(self, name, fn, fn_signature):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    def run(self, **kwargs):
        return self.fn(**kwargs)


def tool(fn: Callable) -> Tool:
    """Decorator: turn a plain function into a Tool (its name, docstring, and typed parameters)."""
    sig = get_fn_signature(fn)
    return Tool(name=sig["name"], fn=fn, fn_signature=json.dumps(sig))


# ---------------------------------------------------------------- the ReAct agent
REACT_SYSTEM_PROMPT = """
## Role
You are a function-calling AI model that reasons and acts in a loop until you can deliver a final response to the user.

## Loop Structure
Each iteration of your loop consists of three steps:

1. **Thought** - Reason about the current situation and decide what to do next. Wrap in <thought></thought> tags.
2. **Action** - Call one tool by emitting a <tool_call> JSON object (see format below).
3. **Observation** - The tool result will be returned wrapped in <observation></observation> tags. Use it to inform your next thought.

Repeat this loop until you have enough information to give a final answer, then wrap it in <response></response> tags.

## Tool Call Format
<tool_call>
{"name": <function-name>, "arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

- Do NOT make assumptions about parameter values.
- Pay close attention to each parameter's `type` and use the correct Python types.
- Use exactly one tool call per turn, and increment `id` for each call.

## Available Tools
<tools>
%s
</tools>

## Example Session
<question>What's the current temperature in Madrid?</question>
<thought>I need to retrieve the current weather in Madrid.</thought>
<tool_call>{"name": "get_current_weather", "arguments": {"location": "Madrid", "unit": "celsius"}, "id": 0}</tool_call>
<observation>{0: {"temperature": 25, "unit": "celsius"}}</observation>
<thought>I now have the temperature for Madrid. I can respond to the user.</thought>
<response>The current temperature in Madrid is 25 degrees Celsius.</response>

## Additional Constraints
- Do not skip directly to a final answer while important evidence is still missing.
- Keep the final answer evidence-bounded: state only what the observations support.
- Never leave the loop without eventually producing a <response>.
"""


class ReactAgent:
    """Reason -> Action -> Observation loop. The model thinks, calls one tool, reads the result,
    and repeats until it emits a <response>."""

    def __init__(self, tools, model="openai/gpt-oss-120b", client=None):
        if client is None:
            raise ValueError("ReactAgent requires a `client` (chat.completions.create).")
        self.client = client
        self.model = model
        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {t.name: t for t in self.tools}

    def add_tool_signatures(self) -> str:
        return "".join(t.fn_signature for t in self.tools)

    def process_tool_calls(self, tool_calls_content: list) -> dict:
        observations = {}
        for tool_call_str in tool_calls_content:
            try:
                tool_call = json.loads(tool_call_str)
            except json.JSONDecodeError:
                print(f"\nSkipping malformed tool call: {tool_call_str}")
                continue
            tool_name = tool_call.get("name") if isinstance(tool_call, dict) else None
            t = self.tools_dict.get(tool_name)
            if t is None:
                print(f"\nSkipping unknown tool request: {tool_name}")
                observations[len(observations)] = f"Unknown tool {tool_name!r}. Use one of: {', '.join(self.tools_dict)}."
                continue
            print(f"\nUsing Tool: {tool_name}")
            tool_call.setdefault("arguments", {})
            try:
                validated = validate_arguments(tool_call, json.loads(t.fn_signature))
                print(f"Tool call: {validated}")
                result = t.run(**validated["arguments"])
            except Exception as e:   # a wrong or missing argument becomes an observation, not a crash
                validated, result = tool_call, f"Tool error: {e}"
            print(f"Tool result: {result}")
            observations[validated.get("id", len(observations))] = result
        return observations

    def run(self, user_msg: str, max_rounds: int = 10) -> str:
        system_prompt = REACT_SYSTEM_PROMPT % self.add_tool_signatures()
        chat_history = ChatHistory([
            build_prompt_structure(system_prompt, "system"),
            build_prompt_structure(user_msg, "user", tag="question"),
        ])
        tools_used, pushbacks = 0, 0
        for _ in range(max_rounds):
            completion = completions_create(self.client, chat_history, self.model)

            thought = extract_tag_content(str(completion), "thought")
            tool_calls = extract_tag_content(str(completion), "tool_call")
            response = extract_tag_content(str(completion), "response")
            if not str(completion).strip():       # some models plan the call silently and send nothing
                update_chat_history(chat_history, "Your reply was empty. Write your <thought> and "
                                    "<tool_call> tags as plain text in the reply.", "user")
                continue
            # A <response> only counts when no tool call came with it. A model that writes a tool call,
            # imagines its result, and answers in one go must run the tool and see the REAL result first.
            if response.found and not tool_calls.found:
                if self.tools_dict and tools_used == 0 and pushbacks < 2:
                    # Evidence check: an answer before any tool has run cannot be based on the evidence.
                    pushbacks += 1
                    update_chat_history(chat_history, completion, "assistant")
                    update_chat_history(chat_history, "You have not called any tool yet, so that answer is "
                                        "not based on evidence. Call a tool first with a <tool_call> block.", "user")
                    continue
                return response.content[0]
            if tool_calls.found:
                completion = str(completion).split("<observation>")[0].split("<response>")[0]
            update_chat_history(chat_history, completion, "assistant")

            if thought.found and thought.content:
                print(f"\nThought: {thought.content[0]}")

            if tool_calls.found:
                observations = self.process_tool_calls(tool_calls.content)
                tools_used += len(observations)
                print(f"\nObservations: {observations}")
                update_chat_history(chat_history, f"<observation>{observations}</observation>", "user")

        return completions_create(self.client, chat_history, self.model)

## The Case in One Minute

**Case `RA-2026-009`** - an outreach worker's Samsung Galaxy A54 was left unattended on a supply
table for a short interval. The question is narrow: during that interval, did the phone try to send
an image through Signal, and can we say anything about whether it was delivered?

| UTC time | What the records show |
|---|---|
| 14:10:00 | unattended interval starts (staff observation) |
| 14:15:58 | mobile data drops offline |
| 14:16:11 | Signal image-attachment attempt recorded |
| 14:25:00 | unattended interval ends |
| 14:28:02 | mobile data restored (after the interval) |

The attempt falls **inside** the interval, but connectivity returned **after** it ended. An attempt
is recorded; delivery is not. Keeping those apart is the point of the lab.

## How ReAct Works, and Where the Memory Lives

Each turn of the loop has three visible parts:

- **Thought** (`<thought>`): the model says what it needs to know next, and why.
- **Action** (`<tool_call>`): the model calls exactly one tool.
- **Observation** (`<observation>`): the tool's result is fed back in.

The loop repeats until the model writes a final **`<response>`**. What lets each turn build on the
last is a simple **short-term memory**: a running `chat_history` list that holds the question, every
thought, every tool result, and the model's messages. Nothing is remembered outside that list, so you
can read the whole reasoning trail top to bottom.

## Part A: Walk the ReAct Loop One Turn at a Time

You will run the loop by hand so every Thought, Action, and Observation is visible before you hand
the case to the packaged agent.

### Step A.1: Define the three forensic tools

**Purpose:** each tool exposes one narrow slice of the case, so the loop has to gather evidence step
by step instead of guessing.  
**Output:** an `available_tools` menu the loop can run from.

In [ ]:
# Purpose: define the three tools, each reading one evidence file.
import csv   # csv: reads spreadsheet-style files (rows of comma-separated values)
import json  # json: reads and writes structured data (lists and labelled fields)
from pathlib import Path

data_dir = Path('data')

def read_csv(filename):
    with (data_dir / filename).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

@tool
def get_incident_window() -> str:
    """Return the start and end timestamps for the reported unattended interval."""
    rows = read_csv('incident_window.csv')
    start = next(r for r in rows if r['event'] == 'unattended_interval_start')
    end = next(r for r in rows if r['event'] == 'unattended_interval_end')
    return json.dumps({'start': start['timestamp_utc'], 'end': end['timestamp_utc'],
                       'start_source': start['source'], 'end_source': end['source'],
                       'start_note': start['notes'], 'end_note': end['notes']})

@tool
def get_message_attempt(app: str) -> str:
    """Return the first recorded messaging attachment attempt for the requested app."""
    rows = read_csv('messaging_events.csv')
    match = next((r for r in rows if r['app'].lower() == app.lower()), None)
    if match is None:
        return json.dumps({'error': f'No messaging event found for {app}.'})
    return json.dumps(match)

@tool
def get_network_restore_time() -> str:
    """Return the timestamp when mobile data connectivity was restored."""
    rows = read_csv('network_events.csv')
    restored = next(r for r in rows if r['status'] == 'restored')
    return restored['timestamp_utc']

available_tools = {t.name: t for t in [get_incident_window, get_message_attempt, get_network_restore_time]}
print('Defined tools:', list(available_tools))

### Step A.2: Build the ReAct prompt and a one-turn helper

**Purpose:** fill the tool descriptions into the ReAct system prompt, state the case question (the
guided walkthrough fixes the tool order so you can inspect each turn), start the `chat_history`, and
define `run_single_react_turn`, which runs exactly one Thought -> Action -> Observation cycle.

In [ ]:
# Purpose: assemble the ReAct prompt, the question, the short-term memory, and a one-turn runner.
tools_signature = ',\n'.join(t.fn_signature for t in available_tools.values())
formatted_system_prompt = REACT_SYSTEM_PROMPT % tools_signature

USER_QUESTION = (
    'Determine whether someone tried to send an image through Signal during the reported unattended '
    'interval, and use network restoration evidence to avoid overstating confirmed delivery. '
    'Use exactly one tool call per turn in this order: get_incident_window, then get_message_attempt '
    'for Signal, then get_network_restore_time. After the third observation, give a final answer with: '
    'ReAct step log (one line per tool step); unattended-interval evidence (start and end timestamps); '
    'Signal image-sending timing evidence (attempt timestamp and whether it is inside the interval); '
    'connectivity context and evidence-bounded conclusion (network restore time, then one careful '
    'sentence that does not claim confirmed delivery).'
)

# chat_history is the agent's short-term memory: question, model thoughts, and observations.
chat_history = [
    {'role': 'system', 'content': formatted_system_prompt},
    {'role': 'user', 'content': f'<question>{USER_QUESTION}</question>'},
]

def run_single_react_turn(history):
    # Ask the model for its next Thought + Action given everything so far.
    model_response = client.chat.completions.create(messages=history, model=MODEL).choices[0].message.content
    print(model_response)
    history.append({'role': 'assistant', 'content': model_response})
    # Look for a <tool_call> block; if there is none, the model may be answering.
    extracted = extract_tag_content(model_response, tag='tool_call')
    if not extracted.found:
        return model_response, None, None
    tool_request = json.loads(extracted.content[0])
    observation = available_tools[tool_request['name']].run(**tool_request['arguments'])
    print('\nTool request:', tool_request)
    print('Observation:', observation)
    history.append({'role': 'user', 'content': f'<observation>{observation}</observation>'})
    return model_response, tool_request, observation
print('Prompt and helper ready.')

### Step A.3: Turn 1 - get the unattended interval

**Purpose:** the model should realize it needs the time window first, before it can judge whether the
Signal event falls inside it.  
**What you'll see:** a Thought, a call to `get_incident_window`, and the observation `14:10:00` to
`14:25:00 UTC`.

In [ ]:
model_response_1, tool_request_1, observation_1 = run_single_react_turn(chat_history)

### Step A.4: Turn 2 - check the Signal attempt

**Purpose:** now that the window is known, the model checks whether a Signal image attempt exists and
when.  
**What you'll see:** a call to `get_message_attempt(app="Signal")` and an attempt at `14:16:11 UTC`,
which is inside the interval.

In [ ]:
model_response_2, tool_request_2, observation_2 = run_single_react_turn(chat_history)

### Step A.5: Turn 3 - check when connectivity returned

**Purpose:** the model checks the network-restore time so it does not mistake an attempt for a
delivery.  
**What you'll see:** mobile data restored at `14:28:02 UTC`, which is after the interval ended.

In [ ]:
model_response_3, tool_request_3, observation_3 = run_single_react_turn(chat_history)

### Step A.6: Ask for the evidence-bounded final answer

**Purpose:** with the three observations gathered, the model writes the four-part answer. It must not
claim the image was delivered: the records show an attempt and a later reconnect, not confirmed
delivery.

In [ ]:
final_output = client.chat.completions.create(messages=chat_history, model=MODEL).choices[0].message.content
print(final_output)

## Part B: Run the Same Case with `ReactAgent`

Now hand the same three tools and the same question to the packaged `ReactAgent`. It manages the
reason-act-observe loop for you. Your job shifts from running the loop to judging it: did it gather
the same observations, in a sensible order, and stay evidence-bounded?

In [ ]:
# Purpose: let ReactAgent drive the loop over the same tools and question.
react_agent = ReactAgent(
    tools=[get_incident_window, get_message_attempt, get_network_restore_time],
    client=client, model=MODEL)

print(react_agent.run(user_msg=USER_QUESTION))

## Your Evidence-Bounded Answer

Write your own four-part answer below. Base every line on the observations you collected, not on
general knowledge.

1. **ReAct step log:** one short line per tool step.
2. **Unattended-interval evidence:** the start and end timestamps.
3. **Signal image-sending timing evidence:** the attempt timestamp, and whether it is inside the interval.
4. **Connectivity context and conclusion:** the network restore time, then one careful sentence that
   does **not** claim confirmed delivery.

In [ ]:
answer = {
    'react_step_log': ['1 get_incident_window ...', '2 get_message_attempt ...', '3 get_network_restore_time ...'],
    'unattended_interval': {'start': '...', 'end': '...'},
    'signal_timing': {'attempt_utc': '...', 'inside_interval': None},
    'connectivity_and_conclusion': {'restored_utc': '...', 'conclusion': '...'},
}
for k, v in answer.items():
    print(k, '->', v)

### Compare the two runs

1. Did `ReactAgent` gather the same three observations you did, in a sensible order?
2. Did it avoid claiming the image was **delivered**?
3. If any claim was too strong, how would you revise it?

## Submit

Save the notebook with Part A, Part B, and your answer filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*Case data and the lab concept are adapted from*
*[frankwxu/agentic-AI4-forensics](https://github.com/frankwxu/agentic-AI4-forensics). The ReAct
framework is adapted from [The Neural Maze's agentic-patterns-course](https://github.com/neural-maze/agentic-patterns-course) (MIT License).*